# Transformer Basics: Theory, Math, and Code

This notebook will guide you through the intuition, mathematics, and implementation of the Transformer and self-attention mechanism.

## 1. What is a Transformer?
Transformers are neural network architectures that rely entirely on self-attention to compute representations of input sequences. They are the foundation of modern LLMs.

Encoder: Processes the input sequence to create contextualized representations.

Decoder: Generates the output sequence, attending to both the input (via encoder-decoder attention) and its own previous outputs (via masked self-attention).

Multi-Head Self-Attention: The core mechanism that allows the model to weigh the importance of different tokens in the sequence.

Feed-Forward Networks: Applied to each token’s representation independently.

Positional Encodings: Add information about token positions since Transformers don’t have inherent sequential order.

Layer Normalization and Residual Connections: Stabilize and improve training.



## 2. Self-Attention: Intuition and Math
The self-attention mechanism allows the model to weigh the importance of different tokens for each position.


Intuition Behind Self-Attention
Imagine you’re reading a sentence: “The cat, which was on the mat, jumped.” To understand “cat,” you need to consider its relationship with “mat” and “jumped.” Self-attention lets the model look at all words in the sentence simultaneously and decide which ones are most relevant to each word.
Self-Attention: Each word (token) gets a score for how much it should “pay attention” to every other word. These scores determine how much each word’s representation contributes to the current word’s updated representation.

Multi-Head: Instead of computing attention once, we compute it multiple times (in parallel) with different perspectives (heads), capturing various relationships (e.g., syntactic, semantic).

Why Multi-Head?:
A single attention mechanism might focus on one type of relationship (e.g., nearby words). Multiple heads allow the model to capture different patterns (e.g., one head for grammar, another for meaning).

It’s like having multiple people read the sentence from different angles and combine their insights.



### Self-Attention Formula
Given queries $Q$, keys $K$, and values $V$ (all matrices):
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$




Token Embeddings:
Each word is converted to a vector (e.g., via WordPiece in BERT).

Positional encodings are added to capture word order.

Query, Key, Value:
Think of Q as asking, “Which words are relevant to me?”

K answers, “Here’s what I’m about.”

V provides, “Here’s my actual content.”

Dot Product:
The dot product QKTQ K^TQ K^T
 measures how similar each token’s query is to every token’s key. High similarity means high relevance.

Scaling:
Dividing by dk\sqrt{d_k}\sqrt{d_k}
 prevents the dot products from becoming too large, which could make the softmax too sharp (favoring one token excessively).

Softmax:
Converts raw scores into a probability distribution, ensuring the weights sum to 1.

Weighted Sum:
The attention weights multiply the value vectors, creating a new representation for each token that’s a mix of relevant tokens.

Multi-Head:
Each head learns a different type of relationship (e.g., one head might focus on nearby words, another on verbs).

Concatenating heads combines these perspectives, and WOW_OW_O
 projects them back to the original dimension.



Input Representation

Suppose you have a sequence of tokens (e.g., words), and each token is represented as an embedding vector. Let’s denote the input as a matrix X of shape (batch_size, seq_len, embed_dim).


Linear Projections: Q, K, V
    From the input X, we linearly project it into three matrices:

- Query Q
- Key K
- Value V

    Each of these projections allows the model to focus differently during attention calculation.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [14]:
test_layer_2 = nn.Linear(8,4)
test_layer_2.weight

Parameter containing:
tensor([[ 0.3524, -0.1077,  0.0887, -0.1409,  0.1806,  0.2173,  0.2939,  0.2459],
        [ 0.0024,  0.1660,  0.2958, -0.3296,  0.1760,  0.2180, -0.1760,  0.1808],
        [-0.2347,  0.1765, -0.3363, -0.2322, -0.0945,  0.1377,  0.2514, -0.0108],
        [-0.2791,  0.3231, -0.1053,  0.0288, -0.0915, -0.0402,  0.2563,  0.2732]],
       requires_grad=True)

In [15]:
test_layer_2.bias

Parameter containing:
tensor([0.0847, 0.2901, 0.2499, 0.0419], requires_grad=True)

In [8]:
test_layer = nn.Linear(5,5)

In [55]:
nn.Linear(2,2)(torch.tensor([[ 0.3802,  0.3112],
                    [ 0.0393, -0.1737, ]]))

tensor([[-0.3105,  0.0140],
        [-0.3722,  0.2166]], grad_fn=<AddmmBackward0>)

In [19]:
test_layer.weight

Parameter containing:
tensor([[ 0.3802, -0.1733, -0.4032,  0.3282,  0.0393],
        [ 0.3112, -0.3067, -0.0165, -0.3853, -0.1737],
        [-0.2886,  0.4092, -0.0853, -0.1229, -0.4318],
        [ 0.4350,  0.0183,  0.3038,  0.3571,  0.2564],
        [-0.0451,  0.3087, -0.0042, -0.0439, -0.3769]], requires_grad=True)

In [42]:
test_layer.weight.transpose(-2,-1)

tensor([[ 0.3802,  0.3112, -0.2886,  0.4350, -0.0451],
        [-0.1733, -0.3067,  0.4092,  0.0183,  0.3087],
        [-0.4032, -0.0165, -0.0853,  0.3038, -0.0042],
        [ 0.3282, -0.3853, -0.1229,  0.3571, -0.0439],
        [ 0.0393, -0.1737, -0.4318,  0.2564, -0.3769]],
       grad_fn=<TransposeBackward0>)

In [48]:
wt = torch.tensor([[ 0.3802,  0.3112],
                    [ 0.0393, -0.1737, ]])

wt2 = torch.tensor([[ 0.3802,  0.0393,],
                    [ 0.3112, -0.1737,]])

In [50]:
.3802 * .3802 + .3112*.3112

0.24139748

In [49]:
torch.matmul(wt,wt2 )

tensor([[ 0.2414, -0.0391],
        [-0.0391,  0.0317]])

In [10]:
test_layer.bias

Parameter containing:
tensor([-0.2213, -0.3569,  0.3172, -0.1293,  0.4437], requires_grad=True)

In [65]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,num_heads):
        
        """
        Initialize the multihead attention layer
        Args:
            d_model: Dimension of the input embedding example (512 )
            num_heads: Number of heads used in the architecture generally 8,12,16,24,32,64
        """
        
        super(MultiHeadAttention,self).__init__()
        
        assert d_model%num_heads ==0, "d_model must be divisible by num_heads completely"
        
        self.d_model = d_model # 512 for BERT model
        self.num_heads = num_heads
        self.d_k = d_model//num_heads #Dimension per head
        
        
        self.W_q = nn.Linear(d_model,d_model) # Query Projection
        self.W_k = nn.Linear(d_model,d_model) # Key Projection
        self.W_v = nn.Linear(d_model,d_model) # Value Projection
        self.W_o = nn.Linear(d_model,d_model) # Output Projection
        
        
    
    def get_attention_score(self,Q,K,V):
        """ 
        Compute the scaled dot product attention
        Args:
            Q (torch.tensor): Query Matrix (batch_size, num_heads,seq_len, d_k)
            K (torch.tensor): Query Matrix (batch_size, num_heads,seq_len, d_k)
            V (torch.tensor): Query Matrix (batch_size, num_heads,seq_len, d_k)
        
        Output:
            attention_projection (torch.tensor): Attention projection
            attention_weights (torch.tensor): Attention Probabilities
        """
        
        batch_size = Q.size(0)
        
        # Compute Attention Score Q.K^T / sqrt(d_k) 
        # shape will be (batch_size,num_heads, seq_len, seq_len)
        scores = torch.matmul(Q,K.transpose(-2,-1)) / math.sqrt(self.d_k)
        
        
        # calculate attention_weights
        # shape will be : (batch_size,num_heads,seq_len, seq_len)
        attention_weights = F.softmax(scores,dim=-1)
        
        # Calculate attention projection
        # shape will be: (batch_size,num_heads,seq_len,d_k)
        attention_projection = torch.matmul(attention_weights, V)
        
        return attention_projection, attention_weights
        
        
    def forward(self,X):
        """ 
        Forward Pass for multihead Attention
        Args:
            X (torch.tensor): input embeddings (batch_size, seq_len, d_model)
            
        Output:
            attention_projection (torch.tensor): Attention projection
            attention_weights (torch.tensor): Attention Probabilities

        """
        
        batch_size , seq_len, d_model = X.size()
        
        # Compute Q,K,V
        # shape: (batch_size,seq_len,d_model)
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # Reshape for multihead attention and split d_model in d_k * num_heads
        # Shape: (batch_size, num_heads, seq_len, d_k)
        Q = Q.view(batch_size,seq_len, self.num_heads,self.d_k).transpose(1,2)
        K = K.view(batch_size,seq_len, self.num_heads,self.d_k).transpose(1,2)
        V = V.view(batch_size,seq_len, self.num_heads,self.d_k).transpose(1,2)       
        
        
        # compute scaled dot_product attention for each head
        # shape will be:
        # attention weights: (batch_size,num_heads,seq_len, seq_len)
        # attention projection: (batch_size,num_heads,seq_len, d_k)
        attention_projection, attention_weights = self.get_attention_score(Q,K,V)
        
        # Combine the heads
        attention_projection = attention_projection.transpose(1,2).contiguous().view(batch_size,seq_len,self.d_model)
        
        # Apply it to output projection # this will help in generating 
        # inter head communication for whole embedding which was not happening with just concatenation
        
        attention_projection_ouput = self.W_o(attention_projection)
        
        return attention_projection_ouput, attention_weights
       

In [66]:
# Hyperparameters
d_model = 512  # Embedding dimension
num_heads = 8  # Number of attention heads
seq_len = 10   # Sequence length
batch_size = 2 # Batch size

# Create dummy input (batch_size, seq_len, d_model)
X = torch.rand(batch_size, seq_len, d_model)

# Initialize multi-head attention
mha = MultiHeadAttention(d_model, num_heads)

# Forward pass
output, attention_weights = mha(X)

print("Input shape:", X.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", attention_weights.shape)


Input shape: torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Attention weights shape: torch.Size([2, 8, 10, 10])


# Adding to a Transformer Encoder
To complete the Transformer encoder layer, you’d add:
Feed-Forward Network (FFN): A two-layer MLP applied to each token.

Residual Connections: Add input to output before normalization.

Layer Normalization: Normalize across the embedding dimension.



 Intuition for the Full Encoder
Multi-Head Attention: Captures relationships between tokens (e.g., “cat” attends to “mat”).

Residual Connection: Ensures the model retains the original input, helping with gradient flow.

LayerNorm: Normalizes the output to stabilize training.

FFN: Adds non-linearity and transforms each token’s representation independently.



In [67]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self,d_model,num_heads,d_ff,dropout = 0.1):
        
        """  
        Transformer encoder layer.
        Args:
            d_model (int): Embeddiing dimension
        
        
        """
        super(TransformerEncoderLayer,self).__init__()
        
        self.mha = MultiHeadAttention(d_model,num_heads)
        
        self.ffn = nn.Sequential(
                        nn.Linear(d_model,d_ff),
                        nn.ReLU(),
                        nn.Linear(d_ff,d_model))
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    
    def forward(self,X):
        # Create forward pass
        attention_projection, attention_weights  = self.mha(X)
        X = self.norm1(X + self.dropout(attention_projection))
        
        # Feed forward with residular connecitons
        ffn_output = self.ffn(X)
        X = self.norm2(X+ self.dropout(ffn_output))
        
        return X, attention_projection
        

In [68]:
# Example usage
encoder_layer = TransformerEncoderLayer(d_model=512, num_heads=8, d_ff=2048)
X = torch.rand(2, 10, 512)  # Batch of 2 sequences, length 10, dim 512
output, attn_weights = encoder_layer(X)
print("Encoder output shape:", output.shape)

Encoder output shape: torch.Size([2, 10, 512])
